**Cellule 1 : Chargement des données et des résultats précédents**

In [1]:
#Importation les bibliothéques nécessaires
import pandas as pd
import networkx as nx
import numpy as np
import random

In [2]:
# 1. Données originales et Clusters
df = pd.read_csv("../data/linkedin_cleaned_data.csv")
df_clusters = pd.read_csv("../resultats/affectation_clusters_louvain.csv")

In [3]:
# 2. On recrée le graphe biparti (Employé - Entreprise)
# On suppose que 'current_company_name' est l'entreprise cible
B = nx.Graph()
B.add_nodes_from(df['id'], bipartite=0) # Employés
B.add_nodes_from(df['current_company_name'].unique(), bipartite=1) # Entreprises

edges = [(row['id'], row['current_company_name']) for _, row in df.iterrows()]
B.add_edges_from(edges)

print(f"✅ Graphe biparti prêt : {B.number_of_nodes()} nœuds, {B.number_of_edges()} liens existants.")

✅ Graphe biparti prêt : 1803 nœuds, 1000 liens existants.


**Cellule 2 : Échantillonnage Négatif (Negative Sampling)**

Pour apprendre, le modèle doit voir des cas où l'employé n'est pas dans l'entreprise.

In [5]:
# On récupère tous les employés et toutes les entreprises
all_employees = df['id'].tolist()
all_companies = df['current_company_name'].unique().tolist()

In [6]:
# Liste des liens positifs (Label 1)
positive_samples = edges
labels_pos = [1] * len(positive_samples)

In [7]:
# Génération de liens négatifs (Label 0)
negative_samples = []
while len(negative_samples) < len(positive_samples):
    emp = random.choice(all_employees)
    comp = random.choice(all_companies)
    
    # On vérifie que le lien n'existe pas déjà
    if not B.has_edge(emp, comp):
        negative_samples.append((emp, comp))

labels_neg = [0] * len(negative_samples)

In [8]:
# Fusion des deux listes
all_samples = positive_samples + negative_samples
all_labels = labels_pos + labels_neg

In [9]:
df_ml = pd.DataFrame(all_samples, columns=['employee_id', 'company_name'])
df_ml['target'] = all_labels

print(f"✅ Dataset créé : {len(df_ml)} exemples (50% positifs / 50% négatifs).")

✅ Dataset créé : 2000 exemples (50% positifs / 50% négatifs).


**Cellule 3 : Création des Features (Topologiques, Communauté, Attributs)**

In [12]:
# Préparation des dictionnaires pour un accès rapide
cluster_map = dict(zip(df_clusters['employee_id'].astype(str), df_clusters['cluster_id']))
emp_info = df.set_index('id').to_dict('index')

# On calcule le degré pour chaque nœud (Feature Topologique)
degres = dict(B.degree())

features = []

for _, row in df_ml.iterrows():
    emp_id = row['employee_id']
    comp_name = row['company_name']
    
    # --- 1. Features Topologiques ---
    # Nombre d'entreprises connues de l'employé et taille de l'entreprise
    emp_deg = degres.get(emp_id, 0)
    comp_deg = degres.get(comp_name, 0)
    pref_attachment = emp_deg * comp_deg
    
    # --- 2. Features de Communauté ---
    # On récupère le cluster de l'employé
    emp_cluster = cluster_map.get(str(emp_id), -1)
    
    # --- 3. Features Attributaires (Similarité) ---
    # On va comparer l'employé avec le profil "type" de l'entreprise
    # (Par exemple, est-ce que l'employé est dans la même industrie que l'entreprise ?)
    info = emp_info.get(emp_id, {})
    
    # On cherche si l'employé partage des points communs avec l'entreprise cible 
    # (via les autres employés de cette entreprise)
    comp_employees = [n for n in B.neighbors(comp_name)]
    
    same_location = 0
    same_industry = 0
    
    if comp_employees:
        # On regarde si l'employé est dans la même ville/industrie que la majorité de l'entreprise
        for ce_id in comp_employees[:10]: # On regarde un échantillon pour aller vite
            ce_info = emp_info.get(ce_id, {})
            if info.get('city') == ce_info.get('city'): same_location = 1
            if info.get('industry') == ce_info.get('industry'): same_industry = 1

    features.append({
        'emp_degree': emp_deg,
        'comp_degree': comp_deg,
        'preferential_attachment': pref_attachment,
        'cluster_id': emp_cluster,
        'same_location': same_location,
        'same_industry': same_industry
    })

# Ajout des features au DataFrame
df_features = pd.DataFrame(features)
df_final_ml = pd.concat([df_ml, df_features], axis=1)

print("✅ Features extraites avec succès.")

✅ Features extraites avec succès.


**Cellule 4 : Sauvegarde du Dataset final**

In [13]:
chemin_ml = "../resultats/dataset_prediction_liens.csv"
df_final_ml.to_csv(chemin_ml, index=False)

print(f"💾 Dataset prêt pour le Machine Learning sauvegardé dans : {chemin_ml}")

💾 Dataset prêt pour le Machine Learning sauvegardé dans : ../resultats/dataset_prediction_liens.csv
